# Phase 7: Ablation Study
Thực hiện các thí nghiệm ablation để đánh giá đóng góp của input context, text representation, lexical features, và source cues.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

In [8]:
PHASE2_PATH = Path('output/phase2/viclickbait_eda_features.csv')
PHASE3_DIR = Path('output/phase3')
PHASE7_DIR = Path('output/phase7')
PHASE7_DIR.mkdir(parents=True, exist_ok=True)

df_features = pd.read_csv(PHASE2_PATH)
df_random_split = pd.read_csv(PHASE3_DIR / 'random_stratified_70_10_20.csv')
df_source_split = pd.read_csv(PHASE3_DIR / 'leave_one_source_out.csv')

# Combine lead_paragraph for 7.1
df_features['lead_paragraph'] = df_features['lead_paragraph'].fillna('')
df_features['title_lead'] = df_features['title'] + ' ' + df_features['lead_paragraph']

In [9]:
def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'clickbait_f1': f1_score(y_true, y_pred, pos_label=1),
        'clickbait_precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'clickbait_recall': recall_score(y_true, y_pred, pos_label=1)
    }

In [10]:
class ItemSelector(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key
    def fit(self, x, y=None):
        return self
    def transform(self, data_dict):
        # Trả về numpy array 2D nếu lấy nhiều cột, ngược lại lấy series 1D
        if isinstance(self.key, list):
            return data_dict[self.key].values
        return data_dict[self.key]

def build_ablation_pipeline(text_col='title', vectorizer_type='word', use_lexical=False):
    features = []

    if vectorizer_type in ['word', 'word_char']:
        features.append(('word_tfidf', Pipeline([
            ('selector', ItemSelector(key=text_col)),
            ('tfidf', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, max_df=0.95))
        ])))
    if vectorizer_type in ['char', 'word_char']:
        features.append(('char_tfidf', Pipeline([
            ('selector', ItemSelector(key=text_col)),
            ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_df=0.95))
        ])))

    if use_lexical:
        features.append(('lexical', Pipeline([
            ('selector', ItemSelector(key=['title_char_len', 'title_word_len', 'title_n_question', 'title_n_cb_keywords'])),
            ('scaler', StandardScaler())
        ])))

    pipeline = Pipeline([
        ('union', FeatureUnion(transformer_list=features)),
        ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000))
    ])
    return pipeline

In [11]:
def evaluate_on_random_split(model_name, pipeline, text_col='title', use_lexical=False):
    train_ids = df_random_split[df_random_split['split'] == 'train']['id']
    test_ids = df_random_split[df_random_split['split'] == 'test']['id']

    df_train = df_features[df_features['id'].isin(train_ids)].copy()
    df_test = df_features[df_features['id'].isin(test_ids)].copy()

    y_train = df_train['label']
    y_test = df_test['label']

    # Dùng thẳng pandas DataFrame để truyền vào Pipeline thay vì dictionary
    pipeline.fit(df_train, y_train)
    y_pred = pipeline.predict(df_test)
    metrics = compute_metrics(y_test, y_pred)
    metrics['model'] = model_name
    return metrics

def evaluate_on_source_heldout(model_name, pipeline, text_col='title', use_lexical=False):
    sources = df_source_split['heldout_source'].unique()
    macro_f1s = []
    for src in sources:
        df_src = df_source_split[df_source_split['heldout_source'] == src]
        train_ids = df_src[df_src['split'] == 'train']['id']
        test_ids = df_src[df_src['split'] == 'test']['id']

        df_train = df_features[df_features['id'].isin(train_ids)].copy()
        df_test = df_features[df_features['id'].isin(test_ids)].copy()

        # Dùng thẳng pandas DataFrame để truyền vào Pipeline thay vì dictionary
        pipeline.fit(df_train, df_train['label'])
        y_pred = pipeline.predict(df_test)
        metrics = compute_metrics(df_test['label'], y_pred)
        macro_f1s.append(metrics['macro_f1'])

    return {'model': model_name, 'avg_heldout_macro_f1': np.mean(macro_f1s)}

In [12]:
# Run Experiments 7.1, 7.2, 7.3
results_random = []
results_heldout = []

experiments = [
    {'name': 'baseline_word_title', 'text_col': 'title', 'vec': 'word', 'lex': False},
    {'name': '7.1_word_title_lead', 'text_col': 'title_lead', 'vec': 'word', 'lex': False},
    {'name': '7.2_char_title', 'text_col': 'title', 'vec': 'char', 'lex': False},
    {'name': '7.2_word_char_title', 'text_col': 'title', 'vec': 'word_char', 'lex': False},
    {'name': '7.3_word_title_lexical', 'text_col': 'title', 'vec': 'word', 'lex': True}
]

for exp in experiments:
    print(f"Running {exp['name']}...")
    pipe = build_ablation_pipeline(text_col=exp['text_col'], vectorizer_type=exp['vec'], use_lexical=exp['lex'])
    res_rand = evaluate_on_random_split(exp['name'], pipe, text_col=exp['text_col'], use_lexical=exp['lex'])
    res_held = evaluate_on_source_heldout(exp['name'], pipe, text_col=exp['text_col'], use_lexical=exp['lex'])

    results_random.append(res_rand)
    results_heldout.append(res_held)

Running baseline_word_title...
Running 7.1_word_title_lead...
Running 7.2_char_title...
Running 7.2_word_char_title...
Running 7.3_word_title_lexical...


In [13]:
# 7.4 Without source-heavy cues
print("Running 7.4_word_title_no_source_cues...")
def remove_source_cues(text):
    # Xoá các từ có xu hướng là shortcut của các trang giải trí
    cues = ['sao', 'bất ngờ', 'sốc', 'kinh ngạc', 'ngã ngửa', 'hé lộ', 'lộ diện']
    pattern = re.compile(r'\b(' + '|'.join(cues) + r')\b', re.IGNORECASE)
    return pattern.sub('', str(text))

df_features['title_masked'] = df_features['title'].apply(remove_source_cues)
pipe_masked = build_ablation_pipeline(text_col='title_masked', vectorizer_type='word', use_lexical=False)

res_rand_masked = evaluate_on_random_split('7.4_word_title_no_source_cues', pipe_masked, text_col='title_masked', use_lexical=False)
res_held_masked = evaluate_on_source_heldout('7.4_word_title_no_source_cues', pipe_masked, text_col='title_masked', use_lexical=False)

results_random.append(res_rand_masked)
results_heldout.append(res_held_masked)

Running 7.4_word_title_no_source_cues...


In [14]:
# Save Results
df_rand = pd.DataFrame(results_random)
df_held = pd.DataFrame(results_heldout)

df_rand.to_csv(PHASE7_DIR / 'ablation_random_split.csv', index=False)
df_held.to_csv(PHASE7_DIR / 'ablation_source_heldout.csv', index=False)

print("\n--- Ablation Random Split ---")
print(df_rand[['model', 'macro_f1', 'clickbait_f1']])
print("\n--- Ablation Source Heldout ---")
print(df_held[['model', 'avg_heldout_macro_f1']])
print("\nPhase 7 experiments saved to output/phase7/")


--- Ablation Random Split ---
                           model  macro_f1  clickbait_f1
0            baseline_word_title  0.727268      0.644211
1            7.1_word_title_lead  0.715449      0.623377
2                 7.2_char_title  0.716460      0.628450
3            7.2_word_char_title  0.718720      0.627706
4         7.3_word_title_lexical  0.729153      0.627635
5  7.4_word_title_no_source_cues  0.708064      0.617021

--- Ablation Source Heldout ---
                           model  avg_heldout_macro_f1
0            baseline_word_title              0.711639
1            7.1_word_title_lead              0.715926
2                 7.2_char_title              0.692795
3            7.2_word_char_title              0.710002
4         7.3_word_title_lexical              0.721466
5  7.4_word_title_no_source_cues              0.693773

Phase 7 experiments saved to output/phase7/
